In [ ]:
#!/usr/bin/env python3
"""SVAMITVA Feature Extraction - Full Pipeline / MoPR x IIT Tirupati 2026"""
import os,sys,glob,json,time,argparse,warnings
warnings.filterwarnings("ignore")
import numpy as np
import torch,torch.nn as nn,torch.nn.functional as F,torch.optim as optim
from torch.utils.data import Dataset,DataLoader
from torch.cuda.amp import GradScaler,autocast
from copy import deepcopy
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import rasterio
from rasterio import features as rio_features
from rasterio.windows import Window
import geopandas as gpd
from shapely.geometry import shape,mapping,box as shapely_box
import cv2,matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt,matplotlib.patches as mpatches,seaborn as sns
from sklearn.metrics import confusion_matrix
torch.backends.cudnn.benchmark = True

from transformers import (
    SegformerForSemanticSegmentation,SegformerConfig,
    Mask2FormerForUniversalSegmentation,Mask2FormerConfig,
)

# ============================================================
# SECTION 1 - CONFIG & PATHS
# ============================================================
HPC_BASE = "/nfsshare/users/P126014014"
WORK_DIR = os.path.join(HPC_BASE,"svamitva_project")
CG_DS3   = os.path.join(HPC_BASE,"CG_Training_dataSet_3","Training_dataSet_3")
CG_DS2   = os.path.join(HPC_BASE,"CG_Training_dataSet_2","Training_dataSet_2")
CG_SHP   = os.path.join(HPC_BASE,"shp-file")

def _find_shp_dir(base):
    for root,dirs,files in os.walk(base):
        if any(f.endswith(".shp") for f in files): return root
    return base

PB_DS  = os.path.join(HPC_BASE,"PB_training_dataSet_shp_file","PB_training_dataSet_shp_file")
PB_SHP = _find_shp_dir(PB_DS)
PB_TEST2 = os.path.join(HPC_BASE,"live_demo_2")
PB_TEST3 = os.path.join(HPC_BASE,"PB_live_demo_3")

CFG = dict(
    n_classes=10,
    patch_size=256,
    stride=256,
    batch_size=8,
    grad_accum=1,
    epochs=8,
    lr=8e-5,
    encoder_lr=3e-5,
    weight_decay=1e-4,
    ema_decay=0.999,

    # 🔥 REQUIRED PATHS (MISSING IN YOUR CODE)
    images_dir=os.path.join(WORK_DIR,"data/training/patches/images"),
    masks_dir=os.path.join(WORK_DIR,"data/training/patches/masks"),
    test_dir=os.path.join(WORK_DIR,"data/testing/images"),
    output_dir=os.path.join(WORK_DIR,"outputs"),

    val_villages=["MURDANDA","28996_NADALA"],
    pretrained=True
)
CLASS_NAMES  = ["Background","RCC_Roof","Tiled_Roof","Tin_Roof","Thatched_Roof",
                "Road","Waterbody","Transformer","Tank","Well"]
CLASS_COLORS = [(200,200,200),(220,50,50),(50,180,50),(50,50,220),(220,160,50),
                (100,100,100),(50,150,220),(220,220,50),(180,50,180),(50,220,180)]
CLASS_FREQ   = torch.tensor([0.65,0.08,0.07,0.05,0.04,0.06,0.03,0.007,0.007,0.006])
CLASS_WEIGHTS= 1.0/(CLASS_FREQ+1e-6); CLASS_WEIGHTS=CLASS_WEIGHTS/CLASS_WEIGHTS.sum()*CFG["n_classes"]
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
DTYPE = torch.float16 if torch.cuda.is_bf16_supported() else torch.float16
BUILT_UP_TYPE_MAP={"rcc":1,"tiled":2,"tin":3,"thatched":4,"pucca":1,"kutcha":4}
for _d in [CFG["images_dir"],CFG["masks_dir"],CFG["test_dir"],CFG["output_dir"],
           os.path.join(CFG["output_dir"],"predictions"),
           os.path.join(CFG["output_dir"],"evaluation"),
           os.path.join(CFG["output_dir"],"visualizations")]:
    os.makedirs(_d,exist_ok=True)
print("Device  : "+str(DEVICE))
print("AMP     : "+str(DTYPE))
print("CG_SHP  : "+CG_SHP)
print("PB_SHP  : "+PB_SHP)

# ============================================================
# SECTION 2 - DATA PREPARATION
# ============================================================
def find_tif_recursive(folder,prefix):
    for root,dirs,files in os.walk(folder):
        for f in files:
            if f.upper().startswith(prefix.upper()) and f.lower().endswith((".tif",".tiff")):
                return os.path.join(root,f)
    return None

def convert_ecw_to_tif(ecw_path):
    tif_path=os.path.splitext(ecw_path)[0]+".tif"
    if os.path.exists(tif_path): return tif_path
    print("  Converting ECW: "+os.path.basename(ecw_path))
    ret=os.system('gdal_translate "'+ecw_path+'" "'+tif_path+'"')
    return tif_path if ret==0 and os.path.exists(tif_path) else None

def find_tif_or_ecw(folder,prefix):
    t=find_tif_recursive(folder,prefix)
    if t: return t
    for root,dirs,files in os.walk(folder):
        for f in files:
            if f.upper().startswith(prefix.upper()) and f.lower().endswith(".ecw"):
                return convert_ecw_to_tif(os.path.join(root,f))
    return None

def verify_tif(p):
    try:
        with rasterio.open(p) as s:
            s.read(window=Window(0,0,min(256,s.width),min(256,s.height)))
        return True
    except Exception as e:
        print("   [CORRUPTED] "+os.path.basename(p)+": "+str(e)); return False

def classify_shp(fn):
    fn=fn.lower()
    if "built_up_area_type" in fn: return "built_up_multi"
    if any(w in fn for w in ["built_up","building","structure","house"]): return 1
    if "road" in fn or "bridge" in fn or "railway" in fn: return 5
    if any(w in fn for w in ["water","pond","lake","river","waterbody"]): return 6
    if "transformer" in fn: return 7
    if "tank" in fn or "storage" in fn: return 8
    if "well" in fn: return 9
    return None

def rasterize_shp(gdf,cls,shape,transform):
    if gdf.empty: return np.zeros(shape,np.uint8)
    geoms=[(mapping(g),cls) for g in gdf.geometry if g is not None]
    return rio_features.rasterize(geoms,out_shape=shape,transform=transform,fill=0,dtype=np.uint8) if geoms else np.zeros(shape,np.uint8)

def rasterize_built_up(gdf,shape,transform):
    mask=np.zeros(shape,np.uint8)
    tc=next((c for c in ["type","Type","TYPE","roof_type","CATEGORY","Category","class","Class","STRUCTURE","structure"] if c in gdf.columns),None)
    if tc is None:
        print("   [WARN] No type col. Cols:"+str(list(gdf.columns)[:6])); return rasterize_shp(gdf,1,shape,transform)
    print("   Roof types:"+str(list(gdf[tc].unique())[:8]))
    for rv,sub in gdf.groupby(tc):
        key=str(rv).strip().lower(); cls=next((v for k,v in BUILT_UP_TYPE_MAP.items() if k in key),1)
        mask=np.where(rasterize_shp(sub,cls,shape,transform)>0,rasterize_shp(sub,cls,shape,transform),mask)
    return mask

def build_mask(tif_path,shp_dir):
    try:
        with rasterio.open(tif_path) as s: H,W,tf,crs,bounds=s.height,s.width,s.transform,s.crs,s.bounds
    except Exception as e: print("   [ERR] "+str(e)); return None
    print("   TIF:"+str(W)+"x"+str(H)+"  CRS:EPSG:"+str(crs.to_epsg()))
    combined=np.zeros((H,W),np.uint8)
    shps=[os.path.join(r,f) for r,d,fs in os.walk(shp_dir) for f in fs if f.endswith(".shp")]
    if not shps: print("   [FAIL] No SHPs in "+shp_dir); return combined
    tif_box=shapely_box(bounds.left,bounds.bottom,bounds.right,bounds.top)
    for sp in shps:
        fname=os.path.basename(sp); result=classify_shp(fname)
        if result is None: continue
        try:
            gdf=gpd.read_file(sp)
            if gdf.crs is None: gdf=gdf.set_crs("EPSG:4326")
            gdf=gdf.to_crs(crs); gdf=gdf[gdf.geometry.intersects(tif_box)]
            if gdf.empty: continue
        except Exception as e: print("   [ERR] "+fname+": "+str(e)); continue
        layer=rasterize_built_up(gdf,(H,W),tf) if result=="built_up_multi" else rasterize_shp(gdf,result,(H,W),tf)
        combined=np.where(layer>0,layer,combined)
        u=np.unique(layer[layer>0])
        if len(u): print("   [OK] "+fname.ljust(40)+"cls:"+str(u)+" n:"+str(len(gdf)))
    lbl=int((combined>0).sum()); print("   Labeled:"+str(lbl)+"/"+str(H*W)+" ("+str(round(100.*lbl/(H*W),2))+"%)")
    return combined

def tile_village(tif_path,mask,tag,ps,st,idir,mdir,cap=5000):
    try:
        with rasterio.open(tif_path) as s: H,W,nb=s.height,s.width,s.count
    except Exception as e: print("   [ERR]"+str(e)); return 0
    with rasterio.open(tif_path) as s:
        count=0; est=((H-ps)//st+1)*((W-ps)//st+1)
        print("   Tiling "+str(W)+"x"+str(H)+" est="+str(est)+" cap="+str(cap))
        for y in range(0,H-ps+1,st):
            for x in range(0,W-ps+1,st):
                try: patch=s.read(window=Window(x,y,ps,ps))
                except: continue
                if nb>=3: patch=patch[:3]
                else: patch=np.repeat(patch[:1],3,axis=0)
                if (patch==0).mean()>0.85: continue
                t=tag+"_"+str(y).zfill(5)+"_"+str(x).zfill(5)
                Image.fromarray(patch.transpose(1,2,0).astype(np.uint8)).save(os.path.join(idir,t+".png"))
                Image.fromarray(mask[y:y+ps,x:x+ps]).save(os.path.join(mdir,t+".png"))
                count+=1
                if cap and count>=cap: print("   capped at "+str(cap)); return count
            if y>0 and y%(st*20)==0: print("   "+str(count)+" patches row "+str(y)+"/"+str(H))
    return count

def _viz_patches(idir,mdir,n=6):
    imgs=sorted(glob.glob(os.path.join(idir,"*.png")))[:n]
    if not imgs: return
    fig,axes=plt.subplots(2,len(imgs),figsize=(4*len(imgs),8))
    for i,ip in enumerate(imgs):
        mp=ip.replace(idir,mdir); axes[0,i].imshow(np.array(Image.open(ip))); axes[0,i].axis("off")
        axes[0,i].set_title(os.path.basename(ip)[:18],fontsize=7)
        if os.path.exists(mp):
            msk=np.array(Image.open(mp)); col=np.zeros((*msk.shape,3),np.uint8)
            for c,cl in enumerate(CLASS_COLORS): col[msk==c]=cl
            axes[1,i].imshow(col); axes[1,i].axis("off")
            axes[1,i].set_title("cls:"+str(np.unique(msk).tolist()),fontsize=7)
    out=os.path.join(CFG["output_dir"],"visualizations","sample_patches.png")
    plt.tight_layout(); plt.savefig(out,dpi=100); plt.close(); print("  Patches viz -> "+out)

def run_data_prep():
    print("\nCG_SHP="+CG_SHP+"\nPB_SHP="+PB_SHP)
    print("CG SHPs:"+str([os.path.basename(f) for f in glob.glob(os.path.join(CG_SHP,"**","*.shp"),recursive=True)]))
    print("PB SHPs:"+str([os.path.basename(f) for f in glob.glob(os.path.join(PB_SHP,"**","*.shp"),recursive=True)]))
    def resolve(folder,prefix):
        p=find_tif_or_ecw(folder,prefix)
        print("  "+("OK  " if p else "FAIL")+"  "+prefix.ljust(22)+"  "+(os.path.basename(p) if p else "NOT FOUND"))
        return p
    vraw=[("MURDANDA",CG_DS3,"MURDANDA_450879",CG_SHP),("NAGUL",CG_DS3,"NAGUL_450171",CG_SHP),
          ("SAMLUR",CG_DS3,"SAMLUR_450163",CG_SHP),("BADETUMNAR",HPC_BASE,"BADETUMNAR_450157",CG_SHP),
          ("KUTRU",HPC_BASE,"KUTRU_451189",CG_SHP),("NADALA",PB_DS,"28996_NADALA",PB_SHP),
          ("FATTU_BHILA",PB_DS,"37458",PB_SHP),("BAGGA",PB_DS,"37774",PB_SHP),
          ("PINDORI",PB_DS,"PINDORI",PB_SHP),("TIMMOWAL",PB_DS,"TIMMOWAL_37695",PB_SHP)]
    print("\nResolving TIF paths...")
    VILLAGES=[(tag,resolve(folder,prefix),shp) for tag,folder,prefix,shp in vraw]
    import shutil
    for tf in [PB_TEST2,PB_TEST3]:
        if not os.path.exists(tf): print("  [WARN] Test folder not found: "+tf); continue
        for f in glob.glob(os.path.join(tf,"**","*.tif"),recursive=True)+glob.glob(os.path.join(tf,"**","*.TIF"),recursive=True):
            dst=os.path.join(CFG["test_dir"],os.path.basename(f))
            if not os.path.exists(dst): shutil.copy2(f,dst); print("  Test TIF copied: "+os.path.basename(f))
    total=0
    for tag,tif,shp in VILLAGES:
        print("\n"+"="*60+"\nVillage: "+tag)
        if not tif or not os.path.exists(tif): print("  SKIP - not found"); continue
        if not verify_tif(tif): print("  [SKIP] Corrupted"); continue
        mask=build_mask(tif,shp)
        if mask is None: continue
        u=np.unique(mask); print("  Classes:"+str(u))
        if len(u)<=1: print("  [WARN] Background only")
        with rasterio.open(tif) as _s: _w,_h=_s.width,_s.height
        _st,_cap=(1024,5000) if _w>50000 or _h>50000 else (CFG["stride"],8000)
        n=tile_village(tif,mask,tag,CFG["patch_size"],_st,CFG["images_dir"],CFG["masks_dir"],cap=_cap)
        print("  "+str(n)+" patches"); total+=n
    print("\n"+"="*60+"\nTotal patches: "+str(total))
    _viz_patches(CFG["images_dir"],CFG["masks_dir"])

# ============================================================
# SECTION 3 - MODEL DEFINITIONS
# ============================================================
class ChannelAttention(nn.Module):
    def __init__(self,ch,r=16):
        super().__init__(); mid=max(ch//r,4)
        self.avg=nn.AdaptiveAvgPool2d(1); self.mx=nn.AdaptiveMaxPool2d(1)
        self.fc=nn.Sequential(nn.Linear(ch,mid,bias=False),nn.ReLU(),nn.Linear(mid,ch,bias=False))
        self.sig=nn.Sigmoid()
    def forward(self,x):
        b,c=x.shape[:2]; a=self.sig(self.fc(self.avg(x).view(b,c))+self.fc(self.mx(x).view(b,c)))
        return x*a.view(b,c,1,1)

class SpatialAttention(nn.Module):
    def __init__(self,k=7):
        super().__init__(); self.conv=nn.Conv2d(2,1,k,padding=k//2,bias=False); self.sig=nn.Sigmoid()
    def forward(self,x):
        return x*self.sig(self.conv(torch.cat([x.mean(1,keepdim=True),x.max(1,keepdim=True).values],1)))

class DualAttention(nn.Module):
    def __init__(self,ch):
        super().__init__(); self.ca=ChannelAttention(ch); self.sa=SpatialAttention()
    def forward(self,x): return self.sa(self.ca(x))

class ASPP(nn.Module):
    def __init__(self,ic,oc,rates=(1,6,12,18)):
        super().__init__()
        self.branches=nn.ModuleList([nn.Sequential(nn.Conv2d(ic,oc,3,padding=r,dilation=r,bias=False),nn.BatchNorm2d(oc),nn.ReLU(inplace=True)) for r in rates])
        self.gap = nn.Sequential(
    nn.AdaptiveAvgPool2d(1),
    nn.Conv2d(ic, oc, 1, bias=False),
    nn.GroupNorm(8, oc),
    nn.ReLU(inplace=True)
)
        self.proj=nn.Sequential(nn.Conv2d(oc*(len(rates)+1),oc,1,bias=False),nn.BatchNorm2d(oc),nn.ReLU(inplace=True),nn.Dropout(0.3))
    def forward(self,x):
        f=[b(x) for b in self.branches]; f.append(F.interpolate(self.gap(x),x.shape[2:],mode='bilinear',align_corners=True))
        return self.proj(torch.cat(f,1))

class ConvBnRelu(nn.Module):
    def __init__(self,ic,oc):
        super().__init__(); self.net=nn.Sequential(nn.Conv2d(ic,oc,3,padding=1,bias=False),nn.BatchNorm2d(oc),nn.ReLU(inplace=True))
    def forward(self,x): return self.net(x)

class DownBlock(nn.Module):
    def __init__(self,ic,oc):
        super().__init__(); self.pool=nn.MaxPool2d(2); self.conv=nn.Sequential(ConvBnRelu(ic,oc),ConvBnRelu(oc,oc)); self.attn=DualAttention(oc)
    def forward(self,x): return self.attn(self.conv(self.pool(x)))

class UpBlock(nn.Module):
    def __init__(self,ic,sc,oc):
        super().__init__(); self.up=nn.Upsample(scale_factor=2,mode='bilinear',align_corners=True)
        self.conv=nn.Sequential(ConvBnRelu(ic+sc,oc),ConvBnRelu(oc,oc)); self.attn=DualAttention(oc)
    def forward(self,x,skip):
        x=self.up(x); dy=skip.size(2)-x.size(2); dx=skip.size(3)-x.size(3)
        x=F.pad(x,[dx//2,dx-dx//2,dy//2,dy-dy//2]); return self.attn(self.conv(torch.cat([x,skip],1)))

class DuSAUNetPP(nn.Module):
    def __init__(self,n=10):
        super().__init__()
        self.enc1=nn.Sequential(ConvBnRelu(3,64),ConvBnRelu(64,64))
        self.enc2=DownBlock(64,128); self.enc3=DownBlock(128,256); self.enc4=DownBlock(256,512)
        self.bot=nn.Sequential(nn.MaxPool2d(2),ConvBnRelu(512,1024),ConvBnRelu(1024,1024),nn.Dropout2d(0.3))
        self.aspp=ASPP(1024,1024); self.battn=DualAttention(1024)
        self.up4=UpBlock(1024,512,512); self.up3=UpBlock(512,256,256); self.up2=UpBlock(256,128,128); self.up1=UpBlock(128,64,64)
        self.head=nn.Conv2d(64,n,1); self.aux4=nn.Conv2d(512,n,1); self.aux3=nn.Conv2d(256,n,1)
        for m in self.modules():
            if isinstance(m,nn.Conv2d): nn.init.kaiming_normal_(m.weight)
            elif isinstance(m,nn.BatchNorm2d): nn.init.constant_(m.weight,1); nn.init.constant_(m.bias,0)
    def forward(self,x):
        H,W=x.shape[2:]; e1=self.enc1(x); e2=self.enc2(e1); e3=self.enc3(e2); e4=self.enc4(e3)
        b=self.battn(self.aspp(self.bot(e4))); d4=self.up4(b,e4); d3=self.up3(d4,e3); d2=self.up2(d3,e2); d1=self.up1(d2,e1)
        out=self.head(d1)
        if self.training:
            return out,F.interpolate(self.aux4(d4),(H,W),mode='bilinear',align_corners=True),F.interpolate(self.aux3(d3),(H,W),mode='bilinear',align_corners=True)
        return out

class SegFormerB5(nn.Module):
    def __init__(self,n=10,pretrained=True):
        super().__init__()
        if pretrained:
            self.model=SegformerForSemanticSegmentation.from_pretrained("nvidia/segformer-b5-finetuned-ade-640-640",num_labels=n,ignore_mismatched_sizes=True)
        else:
            cfg=SegformerConfig.from_pretrained("nvidia/mit-b5"); cfg.num_labels=n; self.model=SegformerForSemanticSegmentation(cfg)
    def forward(self,x):
        H,W=x.shape[2:]; return F.interpolate(self.model(pixel_values=x).logits,(H,W),mode='bilinear',align_corners=False)

class Mask2FormerSeg(nn.Module):
    def __init__(self,n=10,pretrained=True):
        super().__init__(); self.n=n
        if pretrained:
            self.model=Mask2FormerForUniversalSegmentation.from_pretrained("facebook/mask2former-swin-base-coco-panoptic",num_labels=n,ignore_mismatched_sizes=True)
        else:
            c=Mask2FormerConfig(); c.num_labels=n; self.model=Mask2FormerForUniversalSegmentation(c)
    def forward(self,x):
        H,W=x.shape[2:]; out=self.model(pixel_values=x)
        masks=F.interpolate(out.masks_queries_logits.sigmoid(),(H,W),mode='bilinear',align_corners=False)
        labels=F.softmax(out.class_queries_logits[...,:self.n],dim=-1)
        return torch.einsum('bqc,bqhw->bchw',labels,masks)

class EnsembleModel(nn.Module):
    def __init__(self,n=10,pretrained=True,weights=(0.45,0.35,0.20)):
        super().__init__(); self.n_classes=n
        self.segformer=SegFormerB5(n,pretrained); self.mask2former=Mask2FormerSeg(n,pretrained); self.unetpp=DuSAUNetPP(n)
        self.set_weights(*weights)
    def set_weights(self,w1,w2,w3): t=w1+w2+w3; self.w=[w1/t,w2/t,w3/t]
    def load_weights(self,seg_path=None,m2f_path=None,unet_path=None,device="cuda"):
        for p,m,nm in [(seg_path,self.segformer,"SegFormer"),(m2f_path,self.mask2former,"Mask2Former"),(unet_path,self.unetpp,"DuSA U-Net++")]:
            if p and os.path.exists(p): m.load_state_dict(torch.load(p,map_location=device),strict=False); print("[OK] "+nm+" loaded")
    def forward(self,x):
        p1=F.softmax(self.segformer(x),dim=1); p2=F.softmax(self.mask2former(x),dim=1)
        u=self.unetpp(x); u=u[0] if isinstance(u,tuple) else u; p3=F.softmax(u,dim=1)
        return self.w[0]*p1+self.w[1]*p2+self.w[2]*p3
    def forward_individual(self,x):
        self.eval()
        with torch.no_grad():
            p1=F.softmax(self.segformer(x),dim=1); p2=F.softmax(self.mask2former(x),dim=1)
            u=self.unetpp(x); u=u[0] if isinstance(u,tuple) else u; p3=F.softmax(u,dim=1)
        return p1,p2,p3

# ============================================================
# SECTION 4 - LOSS FUNCTIONS (all float32 to avoid autocast errors)
# ============================================================
class LabelSmoothingCE(nn.Module):
    def __init__(self,n,smoothing=0.1,weight=None):
        super().__init__(); self.n=n; self.sm=smoothing; self.w=weight
    def forward(self,pred,target):
        pred=pred.float(); lp=F.log_softmax(pred,dim=1)
        with torch.no_grad():
            td=torch.zeros_like(pred).fill_(self.sm/(self.n-1)); td.scatter_(1,target.unsqueeze(1),1.0-self.sm)
        loss=(-td*lp).sum(1)
        if self.w is not None: loss=loss*self.w.to(pred.device)[target]
        return loss.mean()

class DiceLoss(nn.Module):
    def forward(self,logits,targets):
        n=logits.size(1); p=F.softmax(logits.float(),1)
        t=F.one_hot(targets.clamp(0,n-1),n).permute(0,3,1,2).float()
        inter=(p*t).sum((0,2,3)); union=p.sum((0,2,3))+t.sum((0,2,3))
        return 1-((2*inter+1)/(union+1)).mean()

class FocalLoss(nn.Module):
    def __init__(self,gamma=2.0,weight=None): super().__init__(); self.gamma=gamma; self.w=weight
    def forward(self,logits,targets):
        w=self.w.to(logits.device) if self.w is not None else None
        ce=F.cross_entropy(logits.float(),targets,weight=w,reduction='none')
        return (((1-torch.exp(-ce))**self.gamma)*ce).mean()

class BoundaryLoss(nn.Module):
    """Boundary-aware loss. Uses manual log to avoid F.binary_cross_entropy autocast issue."""
    def _boundary(self,x):
        return (F.max_pool2d(x,5,stride=1,padding=2)!=-F.max_pool2d(-x,3,stride=1,padding=1)).float()
    def forward(self,logits,targets):
        n=logits.size(1); p=F.softmax(logits.float(),1)
        t=F.one_hot(targets.clamp(0,n-1),n).permute(0,3,1,2).float(); b=self._boundary(t)
        inp=p.clamp(1e-7,1.0-1e-7)
        # Manual BCE: avoids F.binary_cross_entropy which crashes with bfloat16 autocast
        return (-(b*inp.log()+(1.0-b)*(1.0-inp).log())).mean()

class CombinedLoss(nn.Module):
    def __init__(self,n,weights=None,smoothing=0.1,w_ce=0.3,w_dice=0.4,w_focal=0.2,w_bnd=0.1):
        super().__init__()
        self.ce=LabelSmoothingCE(n,smoothing,weights); self.dc=DiceLoss(); self.fc=FocalLoss(weight=weights); self.bnd=BoundaryLoss()
        self.wc,self.wd,self.wf,self.wb=w_ce,w_dice,w_focal,w_bnd
    def forward(self,p,t): return self.wc*self.ce(p,t)+self.wd*self.dc(p,t)+self.wf*self.fc(p,t)+self.wb*self.bnd(p,t)

# ============================================================
# SECTION 5 - TRAINING
# ============================================================
def get_transforms(aug):
    if aug:
        return A.Compose([
            A.HorizontalFlip(p=0.5),
            A.RandomRotate90(p=0.5),
            A.Normalize(mean=[0.485,0.456,0.406],
                        std=[0.229,0.224,0.225]),
            ToTensorV2()
        ])
    return A.Compose([
        A.Normalize(mean=[0.485,0.456,0.406],
                    std=[0.229,0.224,0.225]),
        ToTensorV2()
    ])

class PatchDataset(Dataset):
    def __init__(self,imgs,msks,aug=True,n=10):
        self.imgs=sorted(imgs); self.msks=sorted(msks); self.tf=get_transforms(aug); self.n=n
        assert len(self.imgs)==len(self.msks),"Mismatch:"+str(len(self.imgs))+" vs "+str(len(self.msks))
    def __len__(self): return len(self.imgs)
    def __getitem__(self,i):
        img=np.array(Image.open(self.imgs[i]).convert('RGB')); msk=np.clip(np.array(Image.open(self.msks[i])),0,self.n-1)
        out=self.tf(image=img,mask=msk); return out['image'],out['mask'].long()

def village_split(idir,mdir,val_vils):
    imgs=sorted(glob.glob(os.path.join(idir,"*.png"))); msks=sorted(glob.glob(os.path.join(mdir,"*.png")))
    tr_i,tr_m,vl_i,vl_m=[],[],[],[]
    for im,mk in zip(imgs,msks):
        name=os.path.basename(im).upper(); is_val=any(v.upper() in name for v in val_vils)
        (vl_i if is_val else tr_i).append(im); (vl_m if is_val else tr_m).append(mk)
    print("Train:"+str(len(tr_i))+"  Val:"+str(len(vl_i))); return tr_i,tr_m,vl_i,vl_m

class EMA:
    def __init__(self,model,decay=0.9998):
        self.decay=decay; self.shadow={k:v.clone().float() for k,v in model.state_dict().items()}
    @torch.no_grad()
    def update(self,model):
        for k,v in model.state_dict().items(): self.shadow[k]=self.decay*self.shadow[k]+(1-self.decay)*v.float()
    def apply(self,model):
        dev=next(model.parameters()).device; model.load_state_dict({k:v.to(dev) for k,v in self.shadow.items()})
    def save(self,path): torch.save(self.shadow,path)

def compute_metrics(pl,tl,n):
    p=torch.cat(pl).view(-1).numpy(); t=torch.cat(tl).view(-1).numpy(); pix=(p==t).mean()*100
    ious=[((p==c)&(t==c)).sum()/max(((p==c)|(t==c)).sum(),1)*100 for c in range(n) if ((p==c)|(t==c)).sum()>0]
    return pix,(np.mean(ious) if ious else 0.0)

def _plot_training_curves(hist,mname):
    fig,axes=plt.subplots(1,3,figsize=(15,4)); ep=list(range(1,len(hist["tr_loss"])+1))
    axes[0].plot(ep,hist["tr_loss"],label="Train"); axes[0].plot(ep,hist["vl_loss"],label="Val")
    axes[0].set_title("Loss - "+mname); axes[0].legend(); axes[0].grid(True)
    axes[1].plot(ep,hist["pix_acc"]); axes[1].set_title("Pixel Accuracy"); axes[1].grid(True)
    axes[2].plot(ep,hist["miou"]); axes[2].set_title("Mean IoU"); axes[2].grid(True)
    plt.tight_layout(); out=os.path.join(CFG["output_dir"],"visualizations","training_"+mname+".png")
    plt.savefig(out,dpi=100); plt.close(); print("  Training curves -> "+out)

def train_model(mname,cfg,device,dtype):
    print("\n"+"="*60+"\nTraining: "+mname.upper()+"\n"+"="*60)
    tr_i,tr_m,vl_i,vl_m=village_split(cfg['images_dir'],cfg['masks_dir'],cfg['val_villages'])
    if len(tr_i)==0: print("[ERROR] No patches. Run --phase data first."); return 0.0
    num_workers = min(4, os.cpu_count())
    tr_dl=DataLoader(PatchDataset(tr_i,tr_m,True,cfg['n_classes']),cfg['batch_size'],shuffle=True,num_workers=num_workers,pin_memory=True,persistent_workers=True)
    vl_dl=DataLoader(PatchDataset(vl_i,vl_m,False,cfg['n_classes']),cfg['batch_size'],shuffle=False,num_workers=num_workers,pin_memory=True,persistent_workers=True)
    cw=CLASS_WEIGHTS.to(device); criterion=CombinedLoss(cfg['n_classes'],cw)
    if mname=="unetpp":
        model=DuSAUNetPP(cfg['n_classes']).to(device); optimizer=optim.AdamW(model.parameters(),lr=1e-4,weight_decay=cfg['weight_decay']); is_unet=True
    elif mname=="segformer":
        model=SegFormerB5(cfg['n_classes'],cfg['pretrained']).to(device)
        enc_p=list(model.model.segformer.parameters()); enc_ids={id(p) for p in enc_p}
        dec_p=[p for p in model.parameters() if id(p) not in enc_ids]
        optimizer=optim.AdamW([{"params":enc_p,"lr":cfg['encoder_lr']},{"params":dec_p,"lr":cfg['lr']}],weight_decay=cfg['weight_decay']); is_unet=False
    else:
        model=Mask2FormerSeg(cfg['n_classes'],cfg['pretrained']).to(device)
        enc_p=list(model.model.model.pixel_level_module.encoder.parameters()); enc_ids={id(p) for p in enc_p}
        dec_p=[p for p in model.parameters() if id(p) not in enc_ids]
        optimizer=optim.AdamW([{"params":enc_p,"lr":cfg['encoder_lr']*0.5},{"params":dec_p,"lr":cfg['lr']*0.7}],weight_decay=cfg['weight_decay']); is_unet=False
    ema=EMA(model,cfg['ema_decay']); scaler=GradScaler(); lrs=[pg['lr'] for pg in optimizer.param_groups]
    scheduler=optim.lr_scheduler.OneCycleLR(optimizer,max_lr=lrs,total_steps=len(tr_dl)*cfg['epochs']//cfg['grad_accum'],pct_start=0.1,anneal_strategy='cos')
    best_miou=0.0; sp=os.path.join(cfg['output_dir'],"best_"+mname+".pth"); ep=os.path.join(cfg['output_dir'],"ema_"+mname+".pth")
    hist={"tr_loss":[],"vl_loss":[],"pix_acc":[],"miou":[]}
    print("Params:"+str(round(sum(p.numel() for p in model.parameters())/1e6,1))+"M  Train:"+str(len(tr_dl.dataset))+"  Val:"+str(len(vl_dl.dataset)))
    print("Ep".rjust(4)+" | "+"TrLoss".rjust(8)+" | "+"VlLoss".rjust(8)+" | "+"PixAcc".rjust(7)+" | "+"mIoU".rjust(7)); print("-"*50)
    for epoch in range(1,cfg['epochs']+1):
        t0=time.time(); model.train(); trl=0.0; optimizer.zero_grad()
        n_steps=len(tr_dl); print_every=max(1,n_steps//10)
        for step,(imgs,msks) in enumerate(tr_dl):
            imgs,msks=imgs.to(device),msks.to(device)
            with autocast(dtype=dtype):
                if is_unet: out,a4,a3=model(imgs); loss=criterion(out,msks)+0.3*criterion(a4,msks)+0.2*criterion(a3,msks)
                else: out=model(imgs); loss=criterion(out,msks)
                loss=loss/cfg['grad_accum']
            scaler.scale(loss).backward(); trl+=loss.item()*cfg['grad_accum']
            if (step+1)%cfg['grad_accum']==0:
                scaler.unscale_(optimizer); torch.nn.utils.clip_grad_norm_(model.parameters(),1.0)
                scaler.step(optimizer); scaler.update(); scheduler.step(); optimizer.zero_grad(); ema.update(model)
            if (step+1)%print_every==0 or step==0:
                cur_loss=trl/max(step+1,1)
                pct=100*(step+1)/n_steps
                print("  Ep"+str(epoch)+" step "+str(step+1)+"/"+str(n_steps)+" ("+str(round(pct,0))+"%)  loss="+str(round(cur_loss,4)),flush=True)
        trl/=len(tr_dl); ema_m=deepcopy(model); ema.apply(ema_m); ema_m.eval(); vll=0.0; allp,allt=[],[]
        with torch.no_grad():
            for imgs,msks in vl_dl:
                imgs,msks=imgs.to(device),msks.to(device)
                with autocast(dtype=dtype):
                    out=ema_m(imgs)
                    if isinstance(out,tuple): out=out[0]
                    vll+=criterion(out,msks).item()
                allp.append(out.argmax(1).cpu()); allt.append(msks.cpu())
        vll/=len(vl_dl); pix,miou=compute_metrics(allp,allt,cfg['n_classes']); elapsed=int(time.time()-t0)
        hist["tr_loss"].append(trl); hist["vl_loss"].append(vll); hist["pix_acc"].append(pix); hist["miou"].append(miou)
        print(str(epoch).rjust(4)+" | "+str(round(trl,4)).rjust(8)+" | "+str(round(vll,4)).rjust(8)+" | "+(str(round(pix,2))+"%").rjust(7)+" | "+(str(round(miou,2))+"%").rjust(7)+"  ["+str(elapsed)+"s]")
        if miou>best_miou: best_miou=miou; torch.save(model.state_dict(),sp); ema.save(ep); print("     [SAVED] best mIoU "+str(round(miou,2))+"%")
        # Periodic checkpoint regardless of mIoU
        if epoch%cfg.get('checkpoint_every',5)==0:
            ckpt=os.path.join(cfg['output_dir'],"ckpt_"+mname+"_ep"+str(epoch).zfill(3)+".pth")
            torch.save(model.state_dict(),ckpt); print("     [CKPT] "+ckpt)
        del ema_m; torch.cuda.empty_cache()
    _plot_training_curves(hist,mname); print("\n[BEST] mIoU ["+mname+"]: "+str(round(best_miou,2))+"%"); return best_miou

def run_training():
    results={}
    for mname in ["unetpp","segformer","mask2former"]:: results[mname]=train_model(mname,CFG,DEVICE,DTYPE)
    print("\n--- Training Summary ---")
    for k,v in results.items(): print("  "+k.ljust(15)+"mIoU: "+str(round(v,2))+"%")

# ============================================================
# SECTION 6 - COMPREHENSIVE EVALUATION
# ============================================================
def _plot_confusion_matrix(cm,title,path):
    cm_n=cm.astype(float)/(cm.sum(1,keepdims=True)+1e-8)
    fig,ax=plt.subplots(figsize=(13,11))
    sns.heatmap(cm_n,annot=True,fmt='.2f',cmap='Blues',xticklabels=CLASS_NAMES,yticklabels=CLASS_NAMES,ax=ax,vmin=0,vmax=1,linewidths=0.3)
    ax.set_xlabel("Predicted",fontsize=12); ax.set_ylabel("True",fontsize=12); ax.set_title(title,fontsize=13)
    plt.xticks(rotation=45,ha='right'); plt.tight_layout(); plt.savefig(path,dpi=150); plt.close()

def _plot_per_class_bars(per,title,path):
    names,ious,f1s,precs,recs=[],[],[],[],[]
    for cn,r in per.items():
        if r['support']==0: continue
        names.append(cn); ious.append(r['iou']); f1s.append(r['f1']); precs.append(r['precision']); recs.append(r['recall'])
    x=np.arange(len(names)); w=0.2; fig,ax=plt.subplots(figsize=(14,6))
    ax.bar(x-1.5*w,ious,w,label='IoU',color='steelblue'); ax.bar(x-0.5*w,f1s,w,label='F1',color='darkorange')
    ax.bar(x+0.5*w,precs,w,label='Prec',color='green'); ax.bar(x+1.5*w,recs,w,label='Rec',color='red')
    ax.set_xticks(x); ax.set_xticklabels(names,rotation=30,ha='right'); ax.set_ylabel("Score (%)"); ax.set_title(title)
    ax.legend(); ax.grid(axis='y',alpha=0.3); ax.set_ylim(0,105); plt.tight_layout(); plt.savefig(path,dpi=120); plt.close()

def _plot_class_distribution(mdir,path):
    counts=np.zeros(CFG['n_classes'],dtype=np.int64)
    for mp in glob.glob(os.path.join(mdir,"*.png")):
        m=np.array(Image.open(mp))
        for c in range(CFG['n_classes']): counts[c]+=(m==c).sum()
    total=counts.sum(); pcts=100.*counts/max(total,1)
    fig,ax=plt.subplots(figsize=(12,5))
    bars=ax.bar(CLASS_NAMES,pcts,color=[tuple(c/255 for c in col) for col in CLASS_COLORS],edgecolor='black',linewidth=0.5)
    for bar,pct in zip(bars,pcts):
        if pct>0.5: ax.text(bar.get_x()+bar.get_width()/2,bar.get_height()+0.2,str(round(pct,1))+"%",ha='center',va='bottom',fontsize=8)
    ax.set_ylabel("Pixel %"); ax.set_title("Class Distribution in Training Masks")
    ax.set_xticklabels(CLASS_NAMES,rotation=30,ha='right'); ax.grid(axis='y',alpha=0.3)
    plt.tight_layout(); plt.savefig(path,dpi=120); plt.close(); print("  Class dist -> "+path)

def _plot_model_comparison(results_all):
    metrics=["miou","mf1","pix_acc"]; labels=["Mean IoU","Mean F1","Pixel Acc"]; models=list(results_all.keys())
    x=np.arange(len(metrics)); w=0.25; fig,ax=plt.subplots(figsize=(10,6))
    for i,m in enumerate(models):
        vals=[results_all[m][k] for k in metrics]; ax.bar(x+i*w,vals,w,label=m)
        for j,v in enumerate(vals): ax.text(x[j]+i*w,v+0.3,str(round(v,1))+"%",ha='center',va='bottom',fontsize=8)
    ax.set_xticks(x+w); ax.set_xticklabels(labels); ax.set_ylabel("Score (%)"); ax.set_title("Model Comparison")
    ax.legend(); ax.set_ylim(0,105); ax.grid(axis='y',alpha=0.3); plt.tight_layout()
    out=os.path.join(CFG['output_dir'],"evaluation","model_comparison.png"); plt.savefig(out,dpi=120); plt.close(); print("  Model comparison -> "+out)

def evaluate_model(model,vl_dl,device,dtype,mname):
    model.eval(); allp,allt=[],[]
    with torch.no_grad():
        for imgs,msks in vl_dl:
            imgs=imgs.to(device)
            with autocast(dtype=dtype):
                out=model(imgs)
                if isinstance(out,tuple): out=out[0]
            allp.append(out.argmax(1).cpu()); allt.append(msks)
    p=torch.cat(allp).view(-1).numpy(); t=torch.cat(allt).view(-1).numpy(); pix=(p==t).mean()*100
    per={}
    for c in range(CFG['n_classes']):
        tp=((p==c)&(t==c)).sum(); fp=((p==c)&(t!=c)).sum(); fn=((p!=c)&(t==c)).sum()
        per[CLASS_NAMES[c]]=dict(precision=float(tp/(tp+fp+1e-8)*100),recall=float(tp/(tp+fn+1e-8)*100),
            f1=float(2*tp/(2*tp+fp+fn+1e-8)*100),iou=float(tp/(tp+fp+fn+1e-8)*100),support=int((t==c).sum()))
    valid=[r for idx,(_,r) in enumerate(per.items()) if idx>0 and r['support']>0]
    miou=np.mean([r['iou'] for r in valid]) if valid else 0.0; mf1=np.mean([r['f1'] for r in valid]) if valid else 0.0
    mprec=np.mean([r['precision'] for r in valid]) if valid else 0.0; mrec=np.mean([r['recall'] for r in valid]) if valid else 0.0
    print("\n-- "+mname+" --")
    print("  Pixel Accuracy : "+str(round(pix,2))+"%")
    print("  Mean IoU       : "+str(round(miou,2))+"%  <- primary metric")
    print("  Mean F1        : "+str(round(mf1,2))+"%")
    print("  Mean Precision : "+str(round(mprec,2))+"%")
    print("  Mean Recall    : "+str(round(mrec,2))+"%")
    print("  "+"Class".ljust(18)+"Prec".rjust(7)+"Rec".rjust(7)+"F1".rjust(7)+"IoU".rjust(7)+"Support".rjust(10)); print("  "+"-"*58)
    for cn,r in per.items():
        if r['support']==0: continue
        print("  "+cn.ljust(18)+(str(round(r['precision'],1))+"%").rjust(7)+(str(round(r['recall'],1))+"%").rjust(7)+(str(round(r['f1'],1))+"%").rjust(7)+(str(round(r['iou'],1))+"%").rjust(7)+str(r['support']).rjust(10))
    cm=confusion_matrix(t,p,labels=list(range(CFG['n_classes'])))
    cmp=os.path.join(CFG['output_dir'],"evaluation","cm_"+mname+".png"); _plot_confusion_matrix(cm,"Confusion Matrix - "+mname,cmp); print("  CM -> "+cmp)
    bp=os.path.join(CFG['output_dir'],"evaluation","bars_"+mname+".png"); _plot_per_class_bars(per,"Per-Class Metrics - "+mname,bp); print("  Bars -> "+bp)
    return {"pix_acc":pix,"miou":miou,"mf1":mf1,"mprec":mprec,"mrec":mrec,"per_class":per}

def run_evaluation():
    _,_,vl_i,vl_m=village_split(CFG['images_dir'],CFG['masks_dir'],CFG['val_villages'])
    if len(vl_i)==0: print("[ERROR] No val patches."); return
    vl_dl=DataLoader(PatchDataset(vl_i,vl_m,False,CFG['n_classes']),8,shuffle=False,num_workers=4,pin_memory=True,persistent_workers=True)
    _plot_class_distribution(CFG['masks_dir'],os.path.join(CFG['output_dir'],"evaluation","class_distribution.png"))
    results_all={}
    for mname,MC,fname in [("unetpp",DuSAUNetPP,"ema_unetpp.pth"),("segformer",SegFormerB5,"ema_segformer.pth"),("mask2former",Mask2FormerSeg,"ema_mask2former.pth")]:
        mpth=os.path.join(CFG['output_dir'],fname)
        if not os.path.exists(mpth): print("Skip "+mname+" - no weights at "+mpth); continue
        model=MC(CFG['n_classes'],False).to(DEVICE); ew=torch.load(mpth,map_location=DEVICE)
        model.load_state_dict({k:v.to(DEVICE) for k,v in ew.items()}); results_all[mname]=evaluate_model(model,vl_dl,DEVICE,DTYPE,mname)
    if len(results_all)>1: _plot_model_comparison(results_all)
    out_json=os.path.join(CFG['output_dir'],"evaluation","results.json")
    with open(out_json,"w") as f: json.dump({k:{m:v for m,v in r.items() if m!="per_class"} for k,r in results_all.items()},f,indent=2)
    print("\n[SAVED] results.json -> "+out_json)

# ============================================================
# SECTION 7 - ENSEMBLE INFERENCE + VISUALIZATION
# ============================================================
def gaussian_map(size):
    sigma=size/4.0; ax=np.arange(size)-size//2; g=np.exp(-ax**2/(2*sigma**2)).astype(np.float32)
    g2=np.outer(g,g); return g2/g2.max()

GAUSS=gaussian_map(CFG['patch_size'])
NORM_MEAN=np.array([0.485,0.456,0.406],np.float32)[:,None,None]
NORM_STD=np.array([0.229,0.224,0.225],np.float32)[:,None,None]
CLASS_CFG={0:{"name":"Background","threshold":0.50,"min_px":0,"regularise":False},1:{"name":"RCC_Roof","threshold":0.38,"min_px":20,"regularise":True},
    2:{"name":"Tiled_Roof","threshold":0.38,"min_px":15,"regularise":True},3:{"name":"Tin_Roof","threshold":0.36,"min_px":12,"regularise":True},
    4:{"name":"Thatched_Roof","threshold":0.36,"min_px":10,"regularise":True},5:{"name":"Road","threshold":0.36,"min_px":30,"regularise":False},
    6:{"name":"Waterbody","threshold":0.42,"min_px":50,"regularise":False},7:{"name":"Transformer","threshold":0.25,"min_px":3,"regularise":False},
    8:{"name":"Tank","threshold":0.27,"min_px":4,"regularise":False},9:{"name":"Well","threshold":0.25,"min_px":3,"regularise":False}}

def tta_predict(model,tensor,device):
    preds=[]
    def _run(t):
        with torch.no_grad():
            out=model(t.to(device))
            if isinstance(out,tuple): out=out[0]
            return F.softmax(out,dim=1)[0].cpu()
    for rot in [0,1]:
        for hf in [False,True]:
            for vf in [False,True]:
                t=tensor.clone()
                if hf: t=torch.flip(t,[3])
                if vf: t=torch.flip(t,[2])
                if rot: t=torch.rot90(t,1,[2,3])
                p=_run(t)
                if rot: p=torch.rot90(p,-1,[1,2])
                if vf: p=torch.flip(p,[1])
                if hf: p=torch.flip(p,[2])
                preds.append(p)
    return torch.stack(preds).mean(0).numpy()

def morph_clean(mask,k=3):
    kr=np.ones((k,k),np.uint8); m=cv2.morphologyEx(mask.astype(np.uint8),cv2.MORPH_OPEN,kr)
    return cv2.morphologyEx(m,cv2.MORPH_CLOSE,kr)

def _seg_to_color(seg):
    c=np.zeros((*seg.shape,3),np.uint8)
    for i,col in enumerate(CLASS_COLORS): c[seg==i]=col
    return c

def _save_prediction_visualization(tif_path,seg_map,prob_sum,base_name):
    with rasterio.open(tif_path) as src:
        scale=min(1.0,1024.0/src.width); ow=int(src.width*scale); oh=int(src.height*scale)
        data=src.read(out_shape=(src.count,oh,ow),resampling=rasterio.enums.Resampling.bilinear)
        rgb=np.clip((data[:3].transpose(1,2,0) if data.shape[0]>=3 else np.repeat(data[:1],3,axis=0).transpose(1,2,0)),0,255).astype(np.uint8)
    seg_s=cv2.resize(seg_map.astype(np.uint8),(ow,oh),interpolation=cv2.INTER_NEAREST)
    conf=cv2.resize(prob_sum[1:].max(axis=0),(ow,oh),interpolation=cv2.INTER_LINEAR)
    fig,axes=plt.subplots(1,3,figsize=(18,6))
    axes[0].imshow(rgb); axes[0].set_title("Input ORI"); axes[0].axis("off")
    axes[1].imshow(_seg_to_color(seg_s)); axes[1].set_title("Predicted Segmentation"); axes[1].axis("off")
    axes[1].legend(handles=[mpatches.Patch(color=[c/255 for c in CLASS_COLORS[i]],label=CLASS_NAMES[i]) for i in range(CFG['n_classes'])],loc='lower right',fontsize=6,ncol=2)
    im=axes[2].imshow(conf,cmap='hot',vmin=0,vmax=1); axes[2].set_title("Max Confidence"); axes[2].axis("off"); plt.colorbar(im,ax=axes[2],fraction=0.046)
    plt.suptitle("Prediction: "+base_name,fontsize=11); plt.tight_layout()
    out=os.path.join(CFG['output_dir'],"visualizations","pred_"+base_name[:50]+".png"); plt.savefig(out,dpi=120); plt.close(); print("  Viz -> "+out)

def predict_village(model,tif_path,output_gpkg,device):
    with rasterio.open(tif_path) as src:
        H,W=src.height,src.width; transform=src.transform; crs=src.crs; nb=src.count
        print("   "+str(W)+"x"+str(H)+"  EPSG:"+str(crs.to_epsg()))
        ps=CFG['patch_size']; st=CFG['stride']
        prob_sum=np.zeros((CFG['n_classes'],H,W),np.float32); wgt_sum=np.zeros((H,W),np.float32)
        ys=list(range(0,max(H-ps+1,1),st)); xs=list(range(0,max(W-ps+1,1),st)); total=len(ys)*len(xs); done=0
        for y in ys:
            for x in xs:
                ph=min(ps,H-y); pw=min(ps,W-x); patch=src.read(window=Window(x,y,pw,ph))
                if nb>=3: patch=patch[:3]
                else: patch=np.repeat(patch[:1],3,axis=0)
                if patch.mean()<2.0: done+=1; continue
                pf=np.zeros((3,ps,ps),np.float32); pf[:,:ph,:pw]=patch.astype(np.float32)/255.0; pf=(pf-NORM_MEAN)/NORM_STD
                probs=tta_predict(model,torch.from_numpy(pf).unsqueeze(0),device)
                probs=probs[:,:ph,:pw]; g=GAUSS[:ph,:pw]
                prob_sum[:,y:y+ph,x:x+pw]+=probs*g; wgt_sum[y:y+ph,x:x+pw]+=g; done+=1
                if done%200==0: print("   "+str(done)+"/"+str(total)+" patches")
    valid=wgt_sum>0; prob_sum[:,valid]/=wgt_sum[valid]
    seg_map=np.zeros((H,W),np.uint8)
    for cls in [6,5,1,2,3,4,7,8,9]:
        c2=CLASS_CFG[cls]; mask=morph_clean((prob_sum[cls]>c2["threshold"]).astype(np.uint8)); seg_map[mask==1]=cls
    base=os.path.splitext(os.path.basename(tif_path))[0]; _save_prediction_visualization(tif_path,seg_map,prob_sum,base)
    print("  Predicted class distribution:")
    for cls in range(1,CFG['n_classes']):
        cnt=int((seg_map==cls).sum())
        if cnt>0: print("    "+CLASS_NAMES[cls].ljust(18)+"pixels: "+str(cnt))
    all_features=[]
    for cls in range(1,CFG['n_classes']):
        c2=CLASS_CFG[cls]; cls_mask=(seg_map==cls).astype(np.uint8)
        if cls_mask.sum()==0: continue
        for gd,val in rio_features.shapes(cls_mask,transform=transform):
            if val!=1: continue
            geom=shape(gd)
            if geom.area<c2["min_px"]: continue
            if c2["regularise"]: geom=geom.simplify(0.3,preserve_topology=True)
            all_features.append({"geometry":geom,"class_id":cls,"class_name":c2["name"],"area_m2":round(geom.area,2),"confidence":float(prob_sum[cls][cls_mask==1].mean())})
    if all_features:
        gdf=gpd.GeoDataFrame(all_features,crs=crs); gdf.to_file(output_gpkg,driver="GPKG")
        print("  GeoPackage: "+str(len(gdf))+" features -> "+output_gpkg)
        for k,v in gdf.groupby("class_name").size().to_dict().items(): print("    "+k.ljust(18)+str(v).rjust(5)+" polygons")
        return len(gdf)
    print("  No features detected"); return 0

def run_prediction():
    weights=(0.45,0.35,0.20); wp=os.path.join(CFG['output_dir'],"best_weights.json")
    if os.path.exists(wp):
        with open(wp) as f: weights=tuple(json.load(f)["weights"])
    ensemble=EnsembleModel(CFG['n_classes'],pretrained=False,weights=weights)
    ensemble.load_weights(seg_path=os.path.join(CFG['output_dir'],"ema_segformer.pth"),m2f_path=os.path.join(CFG['output_dir'],"ema_mask2former.pth"),unet_path=os.path.join(CFG['output_dir'],"ema_unetpp.pth"),device=str(DEVICE))
    ensemble=ensemble.to(DEVICE).eval()
    test_tifs=sorted(glob.glob(os.path.join(CFG['test_dir'],"*.tif"))+glob.glob(os.path.join(CFG['test_dir'],"*.TIF")))
    print("Found "+str(len(test_tifs))+" test villages")
    if len(test_tifs)==0: print("[WARN] No test TIFs in "+CFG['test_dir']); return
    total=0
    for tif in test_tifs:
        base=os.path.splitext(os.path.basename(tif))[0]; out=os.path.join(CFG['output_dir'],"predictions",base+"_ensemble.gpkg")
        print("\n>> "+base)
        try: total+=predict_village(ensemble,tif,out,DEVICE)
        except Exception: import traceback; traceback.print_exc()
    print("\n[DONE] Total features: "+str(total))

# ============================================================
# SECTION 8 - ENSEMBLE WEIGHT TUNING
# ============================================================
def run_tune_weights():
    _,_,vl_i,vl_m=village_split(CFG['images_dir'],CFG['masks_dir'],CFG['val_villages'])
    if len(vl_i)==0: print("[ERROR] No val patches."); return
    vl_dl=DataLoader(PatchDataset(vl_i,vl_m,False,CFG['n_classes']),8,shuffle=False,num_workers=4,pin_memory=True,persistent_workers=True)
    ensemble=EnsembleModel(CFG['n_classes'],pretrained=False)
    ensemble.load_weights(seg_path=os.path.join(CFG['output_dir'],"ema_segformer.pth"),m2f_path=os.path.join(CFG['output_dir'],"ema_mask2former.pth"),unet_path=os.path.join(CFG['output_dir'],"ema_unetpp.pth"),device=str(DEVICE))
    ensemble=ensemble.to(DEVICE).eval(); print("Collecting predictions...")
    all_p1,all_p2,all_p3,all_t=[],[],[],[]
    with torch.no_grad():
        for imgs,msks in vl_dl:
            imgs=imgs.to(DEVICE)
            with autocast(dtype=DTYPE): p1,p2,p3=ensemble.forward_individual(imgs)
            all_p1.append(p1.cpu()); all_p2.append(p2.cpu()); all_p3.append(p3.cpu()); all_t.append(msks)
    p1=torch.cat(all_p1); p2=torch.cat(all_p2); p3=torch.cat(all_p3); gt=torch.cat(all_t)
    best_miou=0.0; best_w=(0.45,0.35,0.20); steps=[round(i*0.05,2) for i in range(1,20)]; print("Grid searching...")
    for w1 in steps:
        for w2 in steps:
            w3=round(1.0-w1-w2,2)
            if w3<=0 or w3>0.9: continue
            fused=(w1*p1+w2*p2+w3*p3).argmax(1).view(-1).numpy(); tgt=gt.view(-1).numpy()
            ious=[((fused==c)&(tgt==c)).sum()/max(((fused==c)|(tgt==c)).sum(),1)*100 for c in range(1,CFG['n_classes']) if (tgt==c).sum()>0]
            miou=np.mean(ious) if ious else 0.0
            if miou>best_miou: best_miou=miou; best_w=(w1,w2,w3)
    print("\n[BEST] weights:"+str(best_w)+"  mIoU:"+str(round(best_miou,2))+"%")
    out_json=os.path.join(CFG['output_dir'],"best_weights.json")
    with open(out_json,"w") as f: json.dump({"weights":list(best_w),"miou":best_miou},f,indent=2)
    print("[SAVED] -> "+out_json)

# ============================================================
# MAIN
# ============================================================
def main(phase):
    # Reduce CUDA memory fragmentation (recommended in OOM error message)
    os.environ.setdefault("PYTORCH_ALLOC_CONF","expandable_segments:True")
    print("\nDevice : "+str(DEVICE))
    if torch.cuda.is_available(): print("GPU    : "+torch.cuda.get_device_name(0)+"\nVRAM   : "+str(round(torch.cuda.get_device_properties(0).total_memory/1e9,1))+" GB")
    print("Phase  : "+phase)
    if phase in ("data","all"): print("\n>>> DATA PREPARATION"); run_data_prep()
    if phase in ("train","all"): print("\n>>> TRAINING"); run_training()
    if phase in ("evaluate","all"): print("\n>>> EVALUATION"); run_evaluation()
    if phase in ("tune","all"): print("\n>>> WEIGHT TUNING"); run_tune_weights()
    if phase in ("predict","all"): print("\n>>> PREDICTION"); run_prediction()

# ============================================================
# PHASE CONTROL
# Jupyter: set PHASE below and run the cell
# Terminal: python3 svamitva_pipeline.py --phase train
# ============================================================
PHASE = "train"  # OPTIONS: data | train | evaluate | predict | tune | all

if __name__ == "__main__":
    if "ipykernel" in sys.modules:
        print("Jupyter mode. PHASE = "+PHASE); main(PHASE)
    else:
        parser=argparse.ArgumentParser()
        parser.add_argument("--phase",choices=["data","train","evaluate","predict","tune","all"],default=PHASE)
        main(parser.parse_args().phase)